# 08 — The Story

**Thesis.** A purchase-verification flag is a convenient but **imperfect** proxy for review trustworthiness. We build a classifier that predicts it from language + behavior, then **stress-test it two ways**: against a real labeled fake-review dataset, and across **two platforms** — Amazon (research data) and **self-collected Steam** reviews. Only some signals survive — and a few would actively *mislead* a platform that trusted the proxy.

*(Headline numbers from the executed notebooks: Amazon Reviews 2023 `Subscription_Boxes` (16,216 reviews) + Hollenbeck ground truth; self-collected Steam (3,500 reviews, 5 games). Re-run 01–09 to refresh.)*

## The question
**Primary:** what linguistic & behavioral signals distinguish verified-purchase from unverified reviews, and can they flag suspicious reviews the verification flag misses?

**Validation:** checked against real fake-review labels, which signals actually survive — and where does the proxy mislead?

**Cross-platform:** do the same signals hold on a *different* platform (Steam), and what does richer platform-specific behavior add?

## Data & the weak proxy
- **Amazon (downloaded research data):** McAuley/UCSD Amazon Reviews 2023, unified 16-column schema, product-metadata join. Proxy = *Verified Purchase* (88% verified here).
- **Steam (self-collected via public API):** Amazon blocks polite scraping (a JS bot wall), so the genuinely self-collected dataset is from Steam's review API — `steam_purchase` is the verified-purchase analog, `received_for_free` flags incentivized reviews, plus rich behavior (author review count, games owned, **playtime**). Joined with `appdetails` metadata.
- **Ground truth:** Hollenbeck et al. (MIT), 381,734 labeled reviews, mapped onto the *same* schema.
- **Proxy honesty:** the flag measures *purchase verification*, **not** deception — the whole point of the validation.

## What we found

**Amazon** (16,216 reviews, 88% verified):

| Result | Value |
|---|---|
| Unverified reviews are **longer** (median words, verified→unverified) | 28 → **53** |
| Trust classifier on the proxy (random forest, ROC-AUC) | **0.80** |
| Reality check on **real** fakes (rating-independent `labeled`, RF ROC-AUC) | **0.72** |
| Same, `primary` label (defines fake via 5-star → circular) | 0.99 ⚠️ |

**Signal survival (proxy vs real label):**
- ✅ **Survives both** — `char_count`, `word_count`: review **length/effort** separates unverified reviews *and* real fakes.
- ⚠️ **Proxy-only (misleading)** — `helpful_votes`, `helpful_votes_pz`, `sentiment_neg`: track the verified flag but **not** real deception.

**Steam** (3,500 self-collected reviews, 5 games):

| Result | Value |
|---|---|
| Trust classifier on the proxy (paid games, RF ROC-AUC) | **0.68** |
| Top signal of an unverified review | **`author_playtime_forever`** |

**Cross-platform takeaway:** review **length** separates low-trust reviews on *both* platforms; beyond that, the most informative signals are **platform-specific behavior** (Steam: playtime & reviewer history; Amazon: helpful-vote patterns), **not** the verified flag itself.

## Figures (poster set)
Exported to `reports/figures/`, embedded in the chapter notebooks.

### Amazon proxy EDA (notebook 02)
![class balance](../reports/figures/02_class_balance.png)
![rating by proxy](../reports/figures/02_rating_by_proxy.png)
![length by proxy](../reports/figures/02_length_by_proxy.png)

### Trust classifier (notebook 04)
![ROC](../reports/figures/04_roc.png)
![feature importances](../reports/figures/04_importances.png)

### Styles & network (notebook 05)
![PCA](../reports/figures/05_pca.png)
![style terms](../reports/figures/05_style_terms.png)

### Ground-truth validation (notebook 07)
![real ROC](../reports/figures/07_real_roc.png)
![signal survival](../reports/figures/07_survival.png)

### Cross-platform: self-collected Steam (notebook 09)
![steam balance](../reports/figures/09_steam_balance.png)
![steam playtime](../reports/figures/09_steam_playtime.png)
![steam ROC](../reports/figures/09_steam_roc.png)
![steam importances](../reports/figures/09_steam_importances.png)

## Real-world impact & honest limitations
**Impact.** A platform could surface *low-trust* reviews the verified flag alone misses — but should weight **review-effort/length** and **reviewer-behavior** signals over helpful-vote or negative-sentiment cues, which the ground-truth check shows are proxy artifacts. The pattern holds across two platforms.

**Limitations (stated plainly):**
- Verified Purchase / `steam_purchase` ≠ deception; it's a *weak proxy*, and ground truth shows where it breaks.
- The `primary` label's 5-star circularity inflates apparent performance — we report the rating-independent `labeled` result as the honest one.
- **Amazon blocks polite scraping** (JS bot wall); circumventing it is out of scope, so self-collected data is from Steam's public API. Steam ratings are binarized recommend/not, free games make the proxy degenerate (handled by restricting to paid games), and `received_for_free` is too rare to model.
- In-scope methods only (no transformers / formal hypothesis tests).

**For the 3-min talk:** question → weak proxy → classifier (~0.80 Amazon) → reality check (~0.72) → *length survives, helpful-votes mislead* → cross-platform on self-collected Steam (~0.68; playtime wins) → the verified flag is useful but imperfect, on **both** platforms.